# I2C device initialization

In [ ]:
%matplotlib tk
from equipment.equipment_init import *
from project.sc8563 import project

if "ic" in globals():
    ic.close()
    
ic = project(device="sc8563", revision="1p1", emulator="ch341", logging=False, is_gui=False)

from phy.relay_16ch import relay_box
relay = relay_box(i2c_h=ic, i2c_a=0x27)

relay.init_channels
relay.logging = False

# test code

In [ ]:
# ic.i2c_scan(update=0)
ic.update_i2c_address(111)

In [ ]:
# iteration for VB FET opeartion

import random

ps821.ch1.disable
ps821.ch2.disable
ps811.disable

delay(1)

ps821.ch1.cfg_all = 0.01, 0.05
ps821.ch2.cfg_all = 5, 0.25
ps811.cfg_all = 0.01, 0.05

delay(1)

ps821.ch2.enable
ps821.ch1.enable
ps811.enable

count = 0
step_0_count = 0 # initial
step_1_count = 0 # off start
step_2_count = 0 # after on
step_3_count_slow_rise = 0 # after ov rise
step_3_count_slow_fall = 0 # after ov fall
step_4_count_init = 0 # init for fast ov
step_4_count_init_count = 0
step_4_count_fast_rise = 0 # after ov rise
step_4_count_fast_fall = 0 # after ov fall
step_5_count = 0 # after off

delay(1)

relay.ch1.enable  # vext1
relay.ch2.disable # vb_out

delay(1)

# for _ in range(1):
while True:

    count += 1

    #----------------------------------------------------------
    #   initialization, default on state
    #----------------------------------------------------------
    vbat = random.randrange(28, 50, 1) / 10
    ps821.ch1.vset = vbat
    ps811.vset = 5
    delay(0.3)

    ic.EXT2_OVP = 1 # 13V
    # ic.EXT2_OVP = 3 # 17V

    step_0_stat = ic.VEXT2_DRV_ON_STAT
    for _ in range(3): step_0_vb_out = round(dm.voltage_100, 1)

    if step_0_vb_out > 4: step_0_count += 1

    #----------------------------------------------------------
    #   manual VB FET off
    #----------------------------------------------------------

    ic.EXT2_SW_CTRL1 = 0
    ic.EXT2_SW_CTRL2 = 0

    delay(0.3)

    step_1_stat = ic.VEXT2_DRV_ON_STAT
    for _ in range(3): step_1_vb_out = round(dm.voltage_100, 1)

    if step_1_vb_out < 1: step_1_count += 1

    #----------------------------------------------------------
    #   manual VB FET on
    #----------------------------------------------------------

    ic.EXT2_SW_CTRL1 = 0
    ic.EXT2_SW_CTRL2 = 1

    delay(0.3)
    step_2_stat = ic.VEXT2_DRV_ON_STAT
    for _ in range(3): step_2_vb_out = round(dm.voltage_100, 1)

    if step_2_vb_out > 4: step_2_count += 1

    #----------------------------------------------------------
    #   slow ov
    #----------------------------------------------------------

    for voltage_rise in range(5, 16, 1):
        ps811.vset = voltage_rise
        delay(0.1)

    for _ in range(3):
        delay(0.2)
        step_3_vb_out_slow_rise = round(dm.voltage_100, 1)
        step_3_stat_slow_rise = ic.VEXT2_DRV_ON_STAT
        print(f"vb_out={step_3_vb_out_slow_rise}, stat={step_3_stat_slow_rise}")
    
    if step_3_vb_out_slow_rise < 3: step_3_count_slow_rise += 1

    for voltage_fall in reversed(range(5, 16, 1)):
        ps811.vset = voltage_fall
        delay(0.07)

    step_3_stat_slow_fall = ic.VEXT2_DRV_ON_STAT
    for _ in range(3): step_3_vb_out_slow_fall = round(dm.voltage_100, 1)

    if step_3_vb_out_slow_fall > 4 : step_3_count_slow_fall += 1

    #----------------------------------------------------------
    #   fast ov
    #----------------------------------------------------------

    relay.ch1.disable
    # relay.ch2.enable  # vb_out charging
    delay(0.3)

    ic.EXT2_SW_CTRL1 = 1

    ps811.vset = 18
    
    delay(0.3)
    step_4_stat_init = ic.VEXT2_DRV_ON_STAT

    if step_4_stat_init == 0 : step_4_count_init_count += 1

    relay.ch1.enable

    delay(0.3)
    step_4_stat_fast_rise = ic.VEXT2_DRV_ON_STAT

    if step_4_stat_fast_rise == 0 : step_4_count_fast_rise += 1

    ps811.vset = 5
    delay(0.3)

    step_4_stat_fast_fall = ic.VEXT2_DRV_ON_STAT
    if step_4_stat_fast_fall == 1 : step_4_count_fast_fall += 1

    # relay.ch2.disable

    #----------------------------------------------------------
    #   manual off
    #----------------------------------------------------------

    ic.EXT2_SW_CTRL1 = 0
    ic.EXT2_SW_CTRL2 = 0

    delay(0.3)
    step_5_stat = ic.VEXT2_DRV_ON_STAT
    for _ in range(3): step_5_vb_out = round(dm.voltage_100, 1)

    if step_5_vb_out < 1 : step_5_count += 1

    log.time_stamp(display=True, ret=False)
    print(f"[{count}, {vbat}V] ", end="")
    print(f"init {step_0_stat}({step_0_count}) ", end="")
    print(f"-> init off {step_1_stat}({step_1_count}) ", end="")
    print(f"-> on {step_2_stat}({step_2_count}) ", end="")
    print(f"-> slow rise {step_3_stat_slow_rise}({step_3_count_slow_rise}) ", end="")
    print(f"-> slow fall {step_3_stat_slow_fall}({step_3_count_slow_fall}) ", end="")
    print(f"-> fast init {step_4_stat_init}({step_4_count_init_count}) ", end="")
    print(f"-> fast rise {step_4_stat_fast_rise}({step_4_count_fast_rise}) ", end="")
    print(f"-> fast fall {step_4_stat_fast_fall}({step_4_count_fast_fall}) ", end="")
    print(f"-> off {step_5_stat}({step_5_count})")

    ps821.ch1.vset = 0.01
    ps811.vset = 0.01
    delay(0.3)